# Orthomosaic — Part 3: Segment (Pixels → Objects with GeoSAM)

**Turn `02_publish`'s orthomosaic COG into georeferenced object polygons — end to end
with GeoBrix's `gbx.models` package.**

This capstone notebook does not run new photogrammetry: it reads the COG the series
already published and shows the two ways to run inference over it:

1. **In-notebook GPU segmentation** — one `runner.segment_raster` call chips the COG,
   runs GeoSAM across the node's GPU(s), stitches label masks across tile seams using
   each tile's own georeferencing, and returns one polygon per detected object.
2. **Served endpoint (Unity Gateway)** — the same GeoSAM backend wrapped as an MLflow
   pyfunc, registered to Unity Catalog, and served behind a GPU Model Serving
   endpoint for programmatic access.

> **Prerequisite.** `02_publish` must have written the orthomosaic COG
> (`orthomosaic_cog.tif`) to its output directory.

> **Runtime.** The in-notebook section needs a **GPU cluster** — the Serverless
> **GPU AI Runtime** (environment 5) or a classic GPU node; GeoSAM's SAM backbone is
> CUDA-bound. The serving section can run afterward from any cluster with `mlflow`
> installed — registration, endpoint creation, and querying are API calls, not
> GPU-bound locally.

---

**Last Update:** September 28, 2026


## Setup


In [ ]:
# Install GeoBrix (GeoSAM GPU deps) for INTERACTIVE runs. Job runs strip %pip and inject
# deps from the environment spec instead (--extras models_gpu_env5 --env-version 5
# --wheel <staged>) -- interactive-only; this cell only affects interactive attach.
%pip install --quiet --disable-pip-version-check --no-deps --force-reinstall "geobrix[light_env5,models_gpu_env5,vizx] @ file:///Volumes/geospatial_docs/geobrix/sample-data/geobrix-0.5.2-py3-none-any.whl"
%pip install --quiet "geobrix[light_env5,models_gpu_env5,vizx] @ file:///Volumes/geospatial_docs/geobrix/sample-data/geobrix-0.5.2-py3-none-any.whl"
dbutils.library.restartPython()

## Configuration


In [ ]:
CATALOG_NAME = "geospatial_docs"
SCHEMA_NAME  = "gopro"

# The published orthomosaic COG from 02_publish, for the series' default group
# ("all" -- config_nb's constant GROUP_KEY_COL="_group" case). See config_nb's
# group_paths()["cog"] for the full per-group path convention; this notebook is
# intentionally standalone (no %run ./config_nb) since it needs the GPU model
# extras, not the photogrammetry extra config_nb's imports assume.
GROUP          = "all"
OUTPUT_DIR     = "/Volumes/geospatial_docs/orthomosaic/data/gopro/outputs/OldOrchard_2017-07-22_result"
ORTHO_COG_PATH = f"{OUTPUT_DIR}/group_{GROUP}/cog/orthomosaic_cog.tif"

# Unity Gateway / GPU serving config -- set PROFILE to your workspace profile.
PROFILE       = "oauth-fe"
MODEL_NAME    = f"{CATALOG_NAME}.{SCHEMA_NAME}.geosam"
ENDPOINT_NAME = "geosam-ortho-endpoint"

## Read the ortho COG

`runner.segment_raster` accepts an open `rasterio` dataset, raw bytes, or a plain
file path — no Spark raster reader needed for a driver-side call. Open it once here
to confirm it is the raster we expect.


In [ ]:
import rasterio

with rasterio.open(ORTHO_COG_PATH) as ds:
    print(
        f"Ortho COG: {ds.width}x{ds.height}px, {ds.count} band(s), "
        f"dtype={ds.dtypes[0]}, crs={ds.crs}"
    )

## Segment the orthomosaic with GeoSAM (in-notebook, GPU)

`runner.segment_raster` is the single GeoBrix call: it chips the COG into an
overlapping grid, runs GeoSAM's automatic-mask segmenter across every visible GPU,
vectorizes each tile's label mask with its own true georeferencing, and merges
fragments that straddle a seam into one polygon per real-world object before
dropping anything under `min_area`. `gpus="all"` probes every visible device; pass
`gpus=1` to force single-GPU execution instead (same result set, different fan-out).


In [ ]:
# GeoSAM's SAM backbone is CUDA-bound -- confirm the attached cluster actually has a
# GPU before spending time chipping a large orthomosaic.
from databricks.labs.gbx import pyrx as rx

_infra = rx.gpu_infra()
assert _infra["gpu_count"] >= 1, (
    "This section needs GPU compute (0 CUDA devices on the driver). Run on the "
    "Serverless GPU AI Runtime (environment 5) or a classic GPU cluster."
)
print(
    f"[nb3] GPU: {_infra['gpu_count']}x, {_infra.get('per_gpu_vram_mb', 0) // 1024}GB VRAM "
    f"each (driver {_infra.get('driver', '?')}, CUDA {_infra.get('cuda', '?')})"
)

In [ ]:
from databricks.labs.gbx.models import runner

# One call: chip -> GeoSAM infer (fanned across every visible GPU) -> stitch across
# tile seams via each tile's own georeferencing -> polygonize -> drop < min_area.
polys = runner.segment_raster(
    ORTHO_COG_PATH,
    model="geosam",
    gpus="all",
    tile_px=1024,
    overlap=64,
    min_area=10.0,
)
print(f"GeoSAM segmented {len(polys)} object(s) (automatic-mask mode)")
polys.head()

# Single-GPU variant -- never imports torch.cuda probing, forces exactly one device
# slot (same result set, different fan-out):
# polys = runner.segment_raster(ORTHO_COG_PATH, model="geosam", gpus=1)

### Preview: segmented objects over the orthomosaic


In [ ]:
import geopandas as gpd
import shapely.wkb

from databricks.labs.gbx import vizx as vz

# polys carries WKB geometry (EPSG:4326, from segment_raster's own georeferencing).
# Wrap it as a GeoDataFrame so vizx's vector_layer can draw it straight from this
# in-memory result -- no intermediate Spark DataFrame needed for a driver-side preview.
polys_gdf = gpd.GeoDataFrame(
    polys.drop(columns=["geom"]),
    geometry=[shapely.wkb.loads(bytes(g)) for g in polys["geom"]],
    crs=4326,
)

vz.plot_static(
    [
        vz.raster_layer(ORTHO_COG_PATH),
        vz.vector_layer(polys_gdf, fill=False, color="red", width=1.2),
    ],
    title=f"GeoSAM segmentation — {len(polys_gdf)} object(s)",
    basemap=False,
)

## Upsize: serve GeoSAM behind a Unity Gateway endpoint

The same GeoSAM backend, wrapped once as an MLflow pyfunc and registered to Unity
Catalog ("Unity Gateway"), becomes a queryable GPU Model Serving endpoint — no
notebook attached, no GPU cluster to keep warm between calls (with
`scale_to_zero=True`). `create_endpoint` is **create-only**: it stands up a new
endpoint and is not the call to roll a new model version onto one that already
exists.


In [ ]:
from databricks.labs.gbx.models import serving

# build_geosam_pyfunc's predict() calls runner.segment_raster server-side, so the
# served model always segments with the same logic exercised above.
pyfunc = serving.build_geosam_pyfunc(model_type="vit_h")

model_version = serving.register_to_unity_gateway(pyfunc, MODEL_NAME, profile=PROFILE)
print(f"Registered {MODEL_NAME} version {model_version}")

In [ ]:
endpoint = serving.create_endpoint(
    MODEL_NAME,
    model_version,
    profile=PROFILE,
    endpoint_name=ENDPOINT_NAME,
    workload_type="GPU_MEDIUM",
    scale_to_zero=True,
)
print(f"Endpoint {ENDPOINT_NAME} ready: {endpoint.get('state')}")

In [ ]:
with open(ORTHO_COG_PATH, "rb") as f:
    tile_bytes = f.read()

# query() base64-encodes the raw bytes and calls the endpoint with the same
# single-string-column shape build_geosam_pyfunc's predict() expects. For a large
# orthomosaic in production, query a representative crop rather than the whole COG
# to bound request size and latency.
result = serving.query(ENDPOINT_NAME, tile_bytes, profile=PROFILE)
print(result["predictions"][0]["geojson"][:500], "...")

## In-notebook GPU vs served endpoint — when to use which

| | In-notebook (`runner.segment_raster`) | Served endpoint (Unity Gateway) |
|---|---|---|
| **Latency** | No network hop; bounded by the attached GPU's own throughput. | An extra request/response round trip on top of inference. |
| **Cold start** | None once the cluster/session is up. | `scale_to_zero=True` pays a cold-start penalty (loading the model onto a GPU) on the first request after idling. |
| **Cost** | Billed for the whole time the interactive GPU cluster is attached, whether or not it is segmenting. | `scale_to_zero=True` bills only while actually serving; `scale_to_zero=False` (provisioned) bills continuously for predictable low latency. |
| **Best for** | Exploration, tuning `tile_px` / `overlap` / `min_area`, one-off or batch runs already inside a notebook or job. | Programmatic or production access — other services, jobs, or apps calling GeoSAM without a notebook attached. |

> **Note.** The characterizations above are expected behavior based on the endpoint
> lifecycle (cold start, scale-to-zero billing), not measured benchmarks for this
> specific model or dataset — actual latency and cost depend on the workload type,
> image size, and traffic pattern.


## Steps performed

**In-notebook segmentation.** `runner.segment_raster` chipped the published
orthomosaic COG, ran GeoSAM (automatic-mask mode) across every visible GPU, stitched
tile-seam fragments via each tile's own georeferencing, and returned one polygon per
detected object — previewed over the orthomosaic with `vizx`.

**Serving.** The same GeoSAM backend was wrapped as an MLflow pyfunc
(`build_geosam_pyfunc`), registered to Unity Catalog (`register_to_unity_gateway`),
stood up behind a GPU Model Serving endpoint (`create_endpoint`,
`scale_to_zero=True`), and queried (`serving.query`).

**Deliverable.** A `pandas.DataFrame` of georeferenced object polygons (`label`,
`geom` WKB, `score`) from the in-notebook run, plus a queryable `ENDPOINT_NAME`
endpoint serving the same model.
